# Build a vector index from a local body of knowledge

Input is a UTF-8 Markdown document. Output is a DuckDB database containing passages and their embeddings. The study used previously extracted SWEBOK Markdown. Corpus files are supplied locally.

For a different document or a corpus with fewer than 101 chunks, see the [working-copy settings and short-document example](../README.md#use-the-notebooks). Choose a new database path when trying a different corpus. The storage cell replaces its target table. The embedding cell sends passages to OpenRouter. The navigable topic tree is prepared separately.


# SWEBOK v4 vectorization → DuckDB

Pipeline:
1. Load `swebok-v4/swebok-v4.md` and strip noise (image lines).
2. Split into chunks with chonkie's [`FastChunker`](https://docs.chonkie.ai/oss/chunkers/fast-chunker).
3. Generate embeddings with `AisuiteEmbeddingModel` (via OpenRouter).
4. Store them in DuckDB in the `swebok_chunks (chunk | embedding)` table with an HNSW index (`vss` extension).
5. Run a test semantic search.

In [ ]:
import re
from pathlib import Path

import duckdb
import numpy as np
from chonkie import FastChunker
from dotenv import load_dotenv
from fast_track_lib.core.embeddings.models import AisuiteEmbeddingModel

load_dotenv()

# repo root (works whether run from experiments/ or the root)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "swebok-v4").exists())
SOURCE = ROOT / "swebok-v4" / "swebok-v4.md"
DB_PATH = ROOT / "swebok.duckdb"
TABLE = "swebok_chunks"

EMBEDDING_MODEL = "openrouter:qwen/qwen3-embedding-8b"
DIM = 2048
CHUNK_SIZE = 2048  # bytes (FastChunker works in bytes, not tokens)

## 1. Load and clean the text

In [2]:
text = SOURCE.read_text(encoding="utf-8")
text = re.sub(r"^!\[image\]\(.*?\)\s*$", "", text, flags=re.M)  # remove image references
text = re.sub(r"\n{3,}", "\n\n", text)                          # collapse blank lines
len(text)

969335

## 2. Chunking with FastChunker

In [3]:
chunker = FastChunker(chunk_size=CHUNK_SIZE, delimiters="\n.?")
chunks = [c.text.strip() for c in chunker.chunk(text)]
chunks = [c for c in chunks if c]

lens = [len(c) for c in chunks]
print(f"{len(chunks)} chunks | mean {np.mean(lens):.0f} chars | min {min(lens)} | max {max(lens)}")
print(chunks[100][:500])

495 chunks | mean 1957 chars | min 215 | max 2048
[Guide preview omitted]


## 3. Embeddings

In [4]:
embedder = AisuiteEmbeddingModel(
    EMBEDDING_MODEL,
    batch_size=32,     # sentences per request
    dimensions=DIM,    # sent with every request to the provider
)

# normalized -> cosine distance == 1 - inner product
embeddings = embedder.encode(chunks, show_progress_bar=True, normalize_embeddings=True)
embeddings = np.asarray(embeddings, dtype=np.float32)
assert embeddings.shape == (len(chunks), DIM), embeddings.shape
embeddings.shape

(495, 2048)

## 4. Store in DuckDB (`chunk | embedding`) with an HNSW index

In [5]:
con = duckdb.connect(str(DB_PATH))
con.execute("INSTALL vss; LOAD vss;")
con.execute("SET hnsw_enable_experimental_persistence = true;")  # required for HNSW on an on-disk database

con.execute(f"DROP TABLE IF EXISTS {TABLE}")
con.execute(f"CREATE TABLE {TABLE} (chunk VARCHAR, embedding FLOAT[{DIM}])")
con.executemany(
    f"INSERT INTO {TABLE} VALUES (?, ?)",
    [(c, e.tolist()) for c, e in zip(chunks, embeddings)],
)
con.execute(f"CREATE INDEX {TABLE}_hnsw ON {TABLE} USING HNSW (embedding) WITH (metric = 'cosine')")

con.sql(f"SELECT count(*) AS n, any_value(len(embedding)) AS dim FROM {TABLE}")

┌───────┬───────┐
│   n   │  dim  │
│ int64 │ int64 │
├───────┼───────┤
│   495 │  2048 │
└───────┴───────┘

## 5. Semantic search

In [6]:
def search(query: str, k: int = 5):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    return con.execute(
        f"""
        SELECT array_cosine_distance(embedding, ?::FLOAT[{DIM}]) AS distance, chunk
        FROM {TABLE}
        ORDER BY distance
        LIMIT ?
        """,
        [np.asarray(q, dtype=np.float32).tolist(), k],
    ).df()

res = search("What is a regression test?")
for _, r in res.iterrows():
    print(f"--- {r.distance:.4f}\n{r.chunk[:300]}\n")

--- 0.2915
[Guide passage omitted]
--- 0.3984
[Guide passage omitted]
--- 0.4197
[Guide passage omitted]
--- 0.4216
[Guide passage omitted]
--- 0.4225
[Guide passage omitted]


In [7]:
# check that the planner uses the HNSW index
probe = ", ".join(["1.0"] * DIM)
plan = con.execute(
    f"EXPLAIN SELECT chunk FROM {TABLE} "
    f"ORDER BY array_cosine_distance(embedding, [{probe}]::FLOAT[{DIM}]) LIMIT 5"
).fetchall()[0][1]
print("uses HNSW:", "HNSW_INDEX_SCAN" in plan)

uses HNSW: True


In [ ]:
con.close()